# Black-Box Optimisation Project
This project tracks and drives an iterative black-box optimisation process across eight independent functions. Each function is evaluated through a black-box platform: an input vector is submitted, and only the resulting output value is returned, the underlying function itself stays hidden. The objective for every function is to find the input that maximises the returned output using as few submissions as possible.

## 1. Environment and weekly results:
Loads the required libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
import warnings
import os
warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

## 2. Weekly Data Entry:
Acts as a weekly log for submitting each round's processed results. Every input paired with its corresponding output keyed by week number.

In [ ]:
# ============================================================
# Weekly Data Entry — Submission History
# ============================================================
# Update protocol:
#   1. Append exactly one new block per week.
#   2. Never edit or remove a prior week's block — the best-value tracker
#      scans the full history, not just the latest entry, so truncating
#      it will corrupt anchor selection for every function.
#   3. LATEST_WEEK and NEXT_WEEK are derived automatically from the
#      highest key present.

WEEKLY_RESULTS = {
    1: {
        1: {"x": [0.054070, 0.324357], "y": 1.77383211376231e-101},
        2: {"x": [0.748431, 0.977621], "y": 0.2805686429410152},
        3: {"x": [0.347859, 0.750073, 0.855229], "y": -0.03243256561820677},
        4: {"x": [0.415836, 0.434072, 0.380664, 0.407748], "y": 0.4724242761718682},
        5: {"x": [0.519929, 0.583747, 0.138939, 0.758860], "y": 2.887110553791017},
        6: {"x": [0.394497, 0.336516, 0.635579, 0.117587, 0.310005], "y": -0.8836858604580006},
        7: {"x": [0.751790, 0.981084, 0.487689, 0.690305, 0.779003, 0.760168], "y": 0.0011695846924702363},
        8: {"x": [0.061403, 0.529888, 0.046147, 0.233406, 0.724306, 0.381791, 0.075956, 0.755671], "y": 9.7746197805799},
    },
    2: {
        1: {"x": [0.731023, 0.733029], "y": 7.622962759210221e-16},
        2: {"x": [0.707476, 0.927029], "y": 0.6348089478904942},
        3: {"x": [0.413867, 0.793714, 0.715778], "y": -0.14426104593425204},
        4: {"x": [0.479151, 0.491668, 0.250838, 0.433797], "y": -3.1415166576222116},
        5: {"x": [0.243831, 0.827313, 0.896885, 0.898502], "y": 1196.5073524628956},
        6: {"x": [0.694176, 0.186349, 0.762627, 0.735911, 0.069728], "y": -0.7017718981444228},
        7: {"x": [0.049477, 0.513045, 0.242472, 0.205099, 0.422422, 0.717963], "y": 1.2615373558221064},
        8: {"x": [0.137114, 0.044647, 0.282198, 0.031686, 0.997369, 0.630585, 0.023010, 0.178610], "y": 9.7857173744755},
    },
    3: {
        1: {"x": [0.100000, 0.100000], "y": 2.244422822053609e-144},
        2: {"x": [0.804132, 0.959434], "y": 0.10418728600575619},
        3: {"x": [0.520344, 0.646230, 0.354167], "y": -0.03717541678640471},
        4: {"x": [0.449847, 0.426466, 0.427067, 0.490178], "y": -1.076684101925188},
        5: {"x": [0.245359, 0.849496, 0.901279, 0.861607], "y": 1133.4761531234917},
        6: {"x": [0.910356, 0.650886, 0.395807, 0.534345, 0.287842], "y": -1.4225212828827627},
        7: {"x": [0.000000, 0.491672, 0.247422, 0.218118, 0.420428, 0.730970], "y": 1.2675798701410441},
        8: {"x": [0.096875, 0.083128, 0.241460, 0.072606, 0.949637, 0.610441, 0.000000, 0.225297], "y": 9.8348157141436},
    },
    4: {
        1: {"x": [0.100000, 0.300000], "y": 1.222575031466043e-83},
        2: {"x": [0.703076, 0.941274], "y": 0.7079787990149211},
        3: {"x": [0.472741, 0.629913, 0.359423], "y": -0.023312044862632362},
        4: {"x": [0.403193, 0.439169, 0.272232, 0.460767], "y": -1.72904818261464},
        5: {"x": [0.263981, 0.866728, 0.921255, 0.879018], "y": 1415.6639116357217},
        6: {"x": [0.688305, 0.194532, 0.705759, 0.724350, 0.032873], "y": -0.6348779016204353},
        7: {"x": [0.025000, 0.491672, 0.247422, 0.218118, 0.420428, 0.757285], "y": 1.26731799883746},
        8: {"x": [0.103098, 0.119763, 0.203472, 0.097325, 0.911992, 0.578841, 0.000000, 0.257574], "y": 9.8758849047854},
    },
    5: {
        1: {"x": [0.900000, 0.900000], "y": 1.4514862479438758e-104},
        2: {"x": [0.720000, 0.960000], "y": 0.5392738507132064},
        3: {"x": [0.480000, 0.650000, 0.300000], "y": -0.06482962883741206},
        4: {"x": [0.420000, 0.450000, 0.400000, 0.420000], "y": 0.14276928953203738},
        5: {"x": [0.250000, 0.880000, 0.930000, 0.890000], "y": 1603.2666382761672},
        6: {"x": [0.720000, 0.180000, 0.740000, 0.750000, 0.020000], "y": -0.6318780503093013},
        7: {"x": [0.030000, 0.500000, 0.250000, 0.220000, 0.430000, 0.790000], "y": 1.1416762379615346},
        8: {"x": [0.110000, 0.120000, 0.200000, 0.100000, 0.940000, 0.600000, 0.000000, 0.260000], "y": 9.87034},
    },
    6: {
        1: {"x": [0.900000, 0.100000], "y": 1.0740340011599467e-231},
        2: {"x": [0.705000, 0.945000], "y": 0.5574117876258713},
        3: {"x": [0.470000, 0.630000, 0.360000], "y": -0.03242613530546181},
        4: {"x": [0.418000, 0.442000, 0.392000, 0.414000], "y": 0.31983675174548454},
        5: {"x": [0.240000, 0.890000, 0.940000, 0.900000], "y": 1793.5476795009736},
        6: {"x": [0.730000, 0.170000, 0.750000, 0.760000, 0.015000], "y": -0.6047136531480874},
        7: {"x": [0.012000, 0.492000, 0.247000, 0.218000, 0.420000, 0.744000], "y": 1.2695159736781236},
        8: {"x": [0.103000, 0.120000, 0.203000, 0.097000, 0.920000, 0.580000, 0.000000, 0.258000], "y": 9.8749896},
    },
    7: {
        1: {"x": [0.100000, 0.900000], "y": 1.0740340011599467e-231},
        2: {"x": [0.704000, 0.943000], "y": 0.7349477412592027},
        3: {"x": [0.471000, 0.631000, 0.358000], "y": -0.02825965261809342},
        4: {"x": [0.416000, 0.440000, 0.388000, 0.410000], "y": 0.33299054044563947},
        5: {"x": [0.235000, 0.895000, 0.945000, 0.905000], "y": 1895.255297747765},
        6: {"x": [0.740000, 0.160000, 0.760000, 0.770000, 0.010000], "y": -0.6786560361723083},
        7: {"x": [0.010000, 0.492000, 0.247000, 0.218000, 0.420000, 0.745000], "y": 1.264286612822888},
        8: {"x": [0.103000, 0.120000, 0.203000, 0.097000, 0.916000, 0.579000, 0.000000, 0.258000], "y": 9.8756206},
    },
    8: {
        1: {"x": [0.500000, 0.500000], "y": 2.6752879910742468e-09},
        2: {"x": [0.704000, 0.944000], "y": 0.594596690880717},
        3: {"x": [0.472000, 0.630000, 0.359000], "y": -0.023530331085411714},
        4: {"x": [0.416000, 0.436000, 0.384000, 0.408000], "y": 0.4210426871908983},
        5: {"x": [0.230000, 0.900000, 0.950000, 0.910000], "y": 2001.5490596699337},
        6: {"x": [0.730000, 0.170000, 0.750000, 0.760000, 0.015000], "y": -0.68939471066643},
        7: {"x": [0.012000, 0.492000, 0.247000, 0.218000, 0.420000, 0.744000], "y": 1.2695159736781236},
        8: {"x": [0.103000, 0.120000, 0.203000, 0.097000, 0.914000, 0.579000, 0.000000, 0.258000], "y": 9.8758506},
    },
    9: {
        1: {"x": [0.520000, 0.540000], "y": 1.5521491396424238e-14},
        2: {"x": [0.704000, 0.943000], "y": 0.5579919343563734},
        3: {"x": [0.472000, 0.630000, 0.359000], "y": -0.030784368400372263},
        4: {"x": [0.448000, 0.427000, 0.425000, 0.488000], "y": -1.0072475656155535},
        5: {"x": [0.225000, 0.905000, 0.955000, 0.915000], "y": 2112.590424719629},
        6: {"x": [0.720000, 0.180000, 0.740000, 0.750000, 0.020000], "y": -0.7082381511126038},
        7: {"x": [0.011000, 0.492000, 0.247000, 0.218000, 0.420000, 0.744000], "y": 1.2677468729956263},
        8: {"x": [0.103098, 0.119763, 0.203472, 0.097325, 0.911992, 0.578841, 0.000000, 0.257574], "y": 9.8758849047854},
    },
    10: {
        1: {"x": [0.493769, 0.529153], "y": 7.752493942567078e-13},
        2: {"x": [0.729396, 0.944006], "y": 0.4838554842249442},
        3: {"x": [0.494668, 0.631894, 0.376880], "y": -0.04035228384637384},
        4: {"x": [0.472321, 0.378599, 0.359116, 0.412260], "y": -0.7647378245413381},
        5: {"x": [0.204072, 0.934342, 0.984813, 0.942418], "y": 2856.8552459597836},
        6: {"x": [0.707725, 0.145886, 0.729310, 0.784187, 0.028533], "y": -0.5439375079184432},
        7: {"x": [0.012000, 0.479672, 0.247422, 0.218118, 0.420428, 0.730970], "y": 1.326559769155365},
        8: {"x": [0.097224, 0.125185, 0.198051, 0.102885, 0.914727, 0.572906, 0.005057, 0.260805], "y": 9.883848865994},
    },
    11: {
        1: {"x": [0.761012, 0.726153], "y": -1.224494572874762e-19},
        2: {"x": [0.707317, 0.936544], "y": 0.49841749694524556},
        3: {"x": [0.534377, 0.647498, 0.365339], "y": -0.03409042884623981},
        4: {"x": [0.431009, 0.406826, 0.446937, 0.500264], "y": -1.2262510484061617},
        5: {"x": [0.180223, 0.964077, 0.999999, 0.971211], "y": 3630.973737651013},
        6: {"x": [0.683398, 0.184777, 0.764662, 0.730222, 0.068633], "y": -0.6423360978386256},
        7: {"x": [0.061441, 0.503246, 0.252987, 0.211734, 0.410422, 0.709379], "y": 1.4032465742660662},
        8: {"x": [0.093308, 0.128800, 0.194437, 0.106592, 0.916550, 0.568949, 0.008429, 0.262959], "y": 9.8888158833999},
    },
    12: {
        1: {"x": [0.513764, 0.515708], "y": 5.570809517514459e-13},
        2: {"x": [0.707240, 0.936557], "y": 0.6018801382487101},
        3: {"x": [0.485020, 0.631022, 0.369199], "y": -0.018773215945616824},
        4: {"x": [0.437953, 0.434601, 0.383141, 0.414769], "y": 0.31511604336182275},
        5: {"x": [0.183413, 0.960641, 0.999999, 0.967718], "y": 3553.2313250840407},
        6: {"x": [0.699103, 0.144628, 0.730938, 0.779636, 0.027657], "y": -0.667303473968618},
        7: {"x": [0.021970, 0.471506, 0.256185, 0.223647, 0.410428, 0.723817], "y": 1.4501925622957117},
        8: {"x": [0.094287, 0.127896, 0.195341, 0.105665, 0.916094, 0.569939, 0.007586, 0.262421], "y": 9.8875995803229},
    },
}

LATEST_WEEK = max(WEEKLY_RESULTS.keys())
FINAL_ANALYSIS_WEEK = LATEST_WEEK
NEW_INPUTS = [WEEKLY_RESULTS[LATEST_WEEK][i]["x"] for i in range(1, 9)]
NEW_OUTPUTS = [WEEKLY_RESULTS[LATEST_WEEK][i]["y"] for i in range(1, 9)]

print(f'Latest week loaded: Week {LATEST_WEEK}')
print(f'Analysis configured through Week {FINAL_ANALYSIS_WEEK}')
print('NEW_INPUTS and NEW_OUTPUTS were generated automatically from WEEKLY_RESULTS.')



# ============================================================
# Best-Value Monitor — Anchor Selection Per Function
# ============================================================
# Rule: every function is maximised.
# For each function, the next-week strategy is anchored to the strongest
# result found in the full submission history so far.

LATEST_WEEK = max(WEEKLY_RESULTS.keys())
TARGET_ANALYSIS_WEEK = LATEST_WEEK

# Reviewed strategy anchors from Week 1–11.
# Weeks displayed on the dashboard.
REVIEWED_BEST_WEEK_FOR_CHART = {
    1: 8,
    2: 7,
    3: 11,
    4: 5,
    5: 11,
    6: 10,
    7: 11,
    8: 11,
}

WEEK12_ANCHOR_LOGIC = {
    1: 'Keep internal historical weak-signal basin; benchmark evidence was weaker',
    2: 'Keep internal Week 7 noisy peak; benchmark evidence was weaker',
    3: 'Use benchmark-informed side-effect basin because benchmark observed a stronger output',
    4: 'Use benchmark-informed placement basin because benchmark slightly improved the best output',
    5: 'Use benchmark boundary/high-yield logic because benchmark result was dramatically stronger',
    6: 'Use benchmark-informed recipe-quality basin because benchmark result was materially stronger',
    7: 'Use benchmark-informed hyperparameter region because benchmark result was materially stronger',
    8: 'Keep internal Week 11 plateau; benchmark evidence was weaker',
}

# Best observed benchmark evidence extracted.
# The notebook compares these against internal best evidence and only adopts benchmark logic where stronger.
BENCHMARK_FINAL_EVIDENCE = {
    1: {'x': [0.759673, 0.752886], 'y': 5.325626e-24, 'source': 'benchmark best visible result'},
    2: {'x': [0.711680, 0.972452], 'y': 0.720820, 'source': 'benchmark best visible result'},
    3: {'x': [0.403000, 0.540200, 0.481400], 'y': -0.000902, 'source': 'benchmark best visible result'},
    4: {'x': [0.428500, 0.428500, 0.428500, 0.428500], 'y': 0.474702, 'source': 'benchmark best visible result'},
    5: {'x': [0.999999, 0.999999, 0.999999, 0.999999], 'y': 8662.405001, 'source': 'benchmark best visible result'},
    6: {'x': [0.422000, 0.356667, 0.552667, 0.748667, 0.160667], 'y': -0.143413, 'source': 'benchmark best visible result'},
    7: {'x': [0.060500, 0.172597, 0.264213, 0.326575, 0.276599, 0.713945], 'y': 2.654559, 'source': 'benchmark best visible result'},
    8: {'x': [0.087620, 0.283766, 0.198043, 0.010183, 0.806732, 0.257449, 0.263252, 0.960983], 'y': 9.868476, 'source': 'benchmark best visible result'},
}

best_monitor_rows = []
BEST_OBSERVED_WEEK = {}
BEST_OBSERVED_VALUE = {}
BEST_OBSERVED_X = {}
BENCHMARK_USED_FOR_FINAL = {}

for fn in range(1, 9):
    fn_records = []
    for week in sorted(WEEKLY_RESULTS.keys()):
        if fn in WEEKLY_RESULTS[week]:
            item = WEEKLY_RESULTS[week][fn]
            fn_records.append((week, float(item['y']), list(item['x'])))

    auto_best_week, auto_best_y, auto_best_x = max(fn_records, key=lambda t: t[1])
    reviewed_week = REVIEWED_BEST_WEEK_FOR_CHART.get(fn, auto_best_week)
    reviewed_matches = [rec for rec in fn_records if rec[0] == reviewed_week]
    if reviewed_matches:
        anchor_week, anchor_y, anchor_x = reviewed_matches[0]
    else:
        anchor_week, anchor_y, anchor_x = auto_best_week, auto_best_y, auto_best_x

    benchmark = BENCHMARK_FINAL_EVIDENCE.get(fn)
    use_benchmark = bool(benchmark and float(benchmark['y']) > float(anchor_y))

    if use_benchmark:
        final_anchor_week = 'Benchmark'
        final_anchor_y = float(benchmark['y'])
        final_anchor_x = list(benchmark['x'])
        final_source = benchmark['source']
    else:
        final_anchor_week = anchor_week
        final_anchor_y = float(anchor_y)
        final_anchor_x = list(anchor_x)
        final_source = 'own reviewed Week 1–11 evidence'

    latest_week, latest_y, latest_x = fn_records[-1]

    BEST_OBSERVED_WEEK[fn] = reviewed_week
    BEST_OBSERVED_VALUE[fn] = final_anchor_y
    BEST_OBSERVED_X[fn] = final_anchor_x
    BENCHMARK_USED_FOR_FINAL[fn] = use_benchmark

    best_monitor_rows.append({
        'Function': f'Function {fn}',
        'Latest Week Available': latest_week,
        'Latest Output': latest_y,
        'Automatic Best Week': auto_best_week,
        'Automatic Best Output': auto_best_y,
        'Reviewed Best Week Shown on Dashboard': reviewed_week,
        'Reviewed Anchor Output': anchor_y,
        'Benchmark Output': None if benchmark is None else float(benchmark['y']),
        'Benchmark Used?': 'YES' if use_benchmark else 'NO',
        'Final Anchor Week': final_anchor_week,
        'Final Anchor Output': final_anchor_y,
        'Final-week Anchor Logic': WEEK12_ANCHOR_LOGIC[fn],
        'Official Direction': 'MAXIMISE',
        'Colour Logic': 'BLUE baseline; GREEN if higher than prior week; RED if lower; accent text marks reviewed best week/anchor',
        'Final Anchor X': '-'.join([f'{float(v):.6f}' for v in final_anchor_x]),
    })

df_best_monitor = pd.DataFrame(best_monitor_rows)
print(f'Latest completed iteration detected: Week {LATEST_WEEK}')
print(f'Analysis configured through the latest completed iteration: Week {TARGET_ANALYSIS_WEEK}')
print('Benchmark evidence used for final functions:', {f: used for f, used in BENCHMARK_USED_FOR_FINAL.items() if used})
display(df_best_monitor)


WEEK_MODE = 'improve'   # 'observe' | 'improve'

## 3. Bayesian optimisation processing:
Fits Gaussian-process models, evaluates completed observations, and prepares final query records based only on available completed weeks.

In [ ]:
BASELINE_CSV = 'path/to/all_functions_combined.csv'
OUTPUT_PATH  = 'path/to/outputs'
X_COLS = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']

os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f'Mode: {WEEK_MODE}')
print(f'Baseline CSV: {BASELINE_CSV}')
print(f'Output path : {OUTPUT_PATH}')




# ============================================================
# Strategy Definitions
# ============================================================
# Next-week strategy after receiving the latest results:
# Use function-specific anchors based on the latest evidence.
# Late-improving functions exploit; regressed functions recover the safer historical basin.

FUNCTION_DESCRIPTIONS = {
    1: "Weak-signal / near-zero detection function: recover around the historically strongest signal.",
    2: "Noisy log-likelihood style function with many local peaks: return to the strongest observed peak.",
    3: "Drug side-effect minimisation transformed into maximisation: return to the most reliable historical basin.",
    4: "Dynamic placement / many local optima: avoid drifting; return to the historically strongest basin.",
    5: "Chemical-process yield with a single dominant peak: continue exploiting the best yield ridge.",
    6: "Recipe quality/residual scoring: return to the strongest stable basin.",
    7: "ML hyperparameter tuning style function: stay near the best observed plateau.",
    8: "High-dimensional complex configuration: micro-refine the best observed basin.",
}

DEFAULT_STRATEGY = {
    1: {'mode': 'ucb', 'beta': 2.0, 'perturb': None, 'note': 'Default broad exploration'},
    2: {'mode': 'ucb', 'beta': 2.0, 'perturb': None, 'note': 'Default broad exploration'},
    3: {'mode': 'ucb', 'beta': 1.5, 'perturb': None, 'note': 'Default moderate exploration'},
    4: {'mode': 'ucb', 'beta': 2.5, 'perturb': None, 'note': 'Default broad exploration'},
    5: {'mode': 'ucb', 'beta': 1.0, 'perturb': None, 'note': 'Default exploitation'},
    6: {'mode': 'ucb', 'beta': 1.5, 'perturb': None, 'note': 'Default moderate exploration'},
    7: {'mode': 'ucb', 'beta': 1.5, 'perturb': None, 'note': 'Default moderate exploration'},
    8: {'mode': 'ucb', 'beta': 2.5, 'perturb': None, 'note': 'Default broad exploration'},
}

# Final-week protection:
# Recalculate anchors directly from WEEKLY_RESULTS.
BEST_BASIN_WEEK_OVERRIDE = dict(BEST_OBSERVED_WEEK) if 'BEST_OBSERVED_WEEK' in globals() else {
    1: 8,
    2: 7,
    3: 11,
    4: 5,
    5: 11,
    6: 10,
    7: 11,
    8: 11,
}

# Hard enforce the reviewed Final-week anchors from the actual Week 11 review.
BEST_BASIN_WEEK_OVERRIDE.update({
    1: 8,
    2: 7,
    3: 11,
    4: 5,
    5: 11,
    6: 10,
    7: 11,
    8: 11,
})

# Exact final query overrides are used only where the attached benchmark evidence is stronger.
FINAL_QUERY_OVERRIDE = {
    fn: np.array(BEST_OBSERVED_X[fn], dtype=float)
    for fn, used in BENCHMARK_USED_FOR_FINAL.items()
    if used
}


IMPROVED_STRATEGY = {
    1: {'mode': 'anchored_exploit', 'beta': 0.18, 'perturb': 0.020, 'xi': 0.000001,
        'note': 'Final-week recovery around historical weak-signal optimum',
        'reason': 'Use the best observed signal as the anchor; avoid chasing a weak latest completed week.'},

    2: {'mode': 'anchored_exploit', 'beta': 0.20, 'perturb': 0.016, 'xi': 0.001,
        'note': 'Final-week recovery around Week 7 noisy peak',
        'reason': 'Noisy function; protect the strongest observed peak with tight local exploitation.'},

    3: {'mode': 'anchored_exploit', 'beta': 0.16, 'perturb': 0.014, 'xi': 0.001,
        'note': 'Final-week exploit Week 11 side-effect basin',
        'reason': 'Week 11 produced the strongest/latest promising side-effect basin; keep refinements local.'},

    4: {'mode': 'anchored_exploit', 'beta': 0.18, 'perturb': 0.018, 'xi': 0.001,
        'note': 'Final-week recover Week 5 placement basin',
        'reason': 'Recover the reviewed Week 5 placement basin and apply controlled local fine-tuning.'},

    5: {'mode': 'anchored_exploit', 'beta': 0.14, 'perturb': 0.018, 'xi': 0.001,
        'trend_follow': True,
        'note': 'Final-week aggressive exploitation around Week 11 yield ridge',
        'reason': 'Best yield ridge should be exploited carefully; trend-follow only if it supports the best basin.'},

    6: {'mode': 'anchored_exploit', 'beta': 0.15, 'perturb': 0.016, 'xi': 0.001,
        'note': 'Final-week hybrid recovery around Week 10 recipe-quality basin',
        'reason': 'Week 11 regressed versus Week 10, so recover the Week 10 basin with controlled exploration.'},

    7: {'mode': 'anchored_exploit', 'beta': 0.14, 'perturb': 0.010, 'xi': 0.001,
        'note': 'Final-week exploit Week 11 hyperparameter region',
        'reason': 'Small local movement only; preserve the best observed plateau.'},

    8: {'mode': 'anchored_exploit', 'beta': 0.08, 'perturb': 0.003, 'xi': 0.001,
        'note': 'Final-week conservative exploitation around Week 11 high-dimensional plateau',
        'reason': 'High-dimensional function is already near a strong plateau; use tiny refinements only.'},
}

ORIGINAL_REASONING = {i: FUNCTION_DESCRIPTIONS[i] for i in range(1, 9)}

print('Strategies defined for weekly strategy monitor.')
print('Benchmark-informed exact overrides enabled:', bool(FINAL_QUERY_OVERRIDE))
print('Best-basin anchors:', BEST_BASIN_WEEK_OVERRIDE)




# ============================================================
# Helper Functions
# ============================================================

def build_history_dataframe(weekly_results):
    """Convert WEEKLY_RESULTS into the same structure as the original baseline CSV."""
    rows = []
    for week, funcs in sorted(weekly_results.items()):
        for fn, item in sorted(funcs.items()):
            row = {
                "week": week,
                "function": f"Function {fn}",
                "output": item["y"],
            }
            for j, val in enumerate(item["x"], start=1):
                row[f"x{j}"] = val
            rows.append(row)
    return pd.DataFrame(rows)


def get_function_history(fn_label, baseline_csv=None):
    """
    Uses BASELINE_CSV if it exists, then appends weekly results.
    """
    hist = build_history_dataframe(WEEKLY_RESULTS)

    if baseline_csv and os.path.exists(baseline_csv):
        try:
            base = pd.read_csv(baseline_csv)
            if "week" not in base.columns:
                base["week"] = 0
            combined = pd.concat([base, hist], ignore_index=True, sort=False)
            combined = combined.drop_duplicates(
                subset=["function"] + X_COLS + ["output"],
                keep="last"
            )
            return combined[combined["function"] == fn_label].copy()
        except Exception as e:
            print(f"  Warning: could not read baseline CSV. Using WEEKLY_RESULTS only. Details: {e}")

    return hist[hist["function"] == fn_label].copy()


def grid_search_f1(X_data, Y_data, n_grid=5):
    """Systematic exploration for Function 1 while output remains near-zero."""
    grid = [[x1, x2]
            for x1 in np.linspace(0.1, 0.9, n_grid)
            for x2 in np.linspace(0.1, 0.9, n_grid)]
    # Add corners/center because F1 has stayed almost zero.
    grid += [[0.1, 0.9], [0.9, 0.1], [0.9, 0.9], [0.5, 0.5]]

    new_grid = []
    for pt in grid:
        already_seen = any(
            np.linalg.norm(np.array(pt) - X_data[j]) < 0.05
            for j in range(len(X_data))
        )
        if not already_seen:
            new_grid.append(pt)

    nonzero_found = any(abs(y) > 1e-10 for y in Y_data)
    print(f'\n  [Function 1 - Grid Mode]')
    if nonzero_found:
        print('  Non-zero output detected — local exploitation can take over next.')
    else:
        print(f'  Still all ~0. {len(new_grid)} candidate grid/corner points remaining.')
    return new_grid[0] if new_grid else [0.5, 0.5]


def anchor_perturb_f7(best_x, n_dims, radius=0.06, x1_range=(0.01, 0.03), x1_target=0.020):
    """Anchor+perturb strategy for Function 7."""
    candidates = []

    for dim in range(1, n_dims):
        for delta in [-radius, -radius/2, -radius/4, radius/4, radius/2, radius]:
            pt = best_x.copy()
            pt[0] = x1_target
            pt[dim] = float(np.clip(pt[dim] + delta, 0, 1))
            candidates.append(pt)

    for x1_val in np.linspace(x1_range[0], x1_range[1], 20):
        pt = best_x.copy()
        pt[0] = x1_val
        candidates.append(pt)

    candidates.append(best_x.copy())
    candidates = np.unique(np.array(candidates), axis=0)
    candidates[:, 0] = np.clip(candidates[:, 0], x1_range[0], x1_range[1])
    return candidates


def format_query(values):
    """Portal-ready format: x1-x2-x3, six decimals."""
    return "-".join([f"{float(v):.6f}" for v in values])


def validate_query(fn_idx, query, expected_dims):
    """Simple safety checks before update."""
    query = np.array(query, dtype=float)
    if len(query) != expected_dims:
        raise ValueError(f"Function {fn_idx}: expected {expected_dims} dims, got {len(query)}.")
    if np.any(query < 0) or np.any(query > 1):
        raise ValueError(f"Function {fn_idx}: query has values outside [0, 1].")
    return np.clip(query, 0, 1)





# ============================================================
# Run the Analysis Update
# ============================================================

missing = [i+1 for i, v in enumerate(NEW_OUTPUTS) if v is None]
if missing:
    print(f'WARNING: Output missing for Function(s): {missing}')
    print('Fill in WEEKLY_RESULTS before running.')
else:
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    strategy = DEFAULT_STRATEGY if WEEK_MODE == 'observe' else IMPROVED_STRATEGY

    comparison_rows = []
    new_query_rows  = []

    for i in range(1, 9):
        # Deterministic per-function randomness for reproducibility.
        rng = np.random.default_rng(4210 + i)
        np.random.seed(4210 + i)

        fn_label = f'Function {i}'
        strat    = strategy[i]

        sub = get_function_history(fn_label, BASELINE_CSV)
        active_cols = [c for c in X_COLS if c in sub.columns and sub[c].notna().any()]

        X_updated = sub[active_cols].values.astype(float)
        Y_updated = sub["output"].values.astype(float)

        n_dims = len(active_cols)
        latest_x = np.array(NEW_INPUTS[i-1], dtype=np.float64).reshape(1, -1)
        latest_y = float(NEW_OUTPUTS[i-1])

        # Prior-week comparator for colour/status logic.
        # Week 1 is baseline and stays neutral. From Week 2 onward: All functions are maximised.
        if len(Y_updated) > 1:
            prior_week_output = float(Y_updated[-2])
            best_before_latest = float(np.max(Y_updated[:-1]))
        else:
            prior_week_output = np.nan
            best_before_latest = float(Y_updated[0])

        best_after = float(np.max(Y_updated))
        best_idx   = int(np.argmax(Y_updated))
        best_x     = X_updated[best_idx].copy()

        # Week 11 final review: allow explicit best-basin anchors by week.
        anchor_week = BEST_BASIN_WEEK_OVERRIDE.get(i, None) if 'BEST_BASIN_WEEK_OVERRIDE' in globals() else None
        if anchor_week is not None and anchor_week in WEEKLY_RESULTS and i in WEEKLY_RESULTS[anchor_week]:
            anchor_x = np.array(WEEKLY_RESULTS[anchor_week][i]['x'], dtype=float)
            anchor_y = float(WEEKLY_RESULTS[anchor_week][i]['y'])
        else:
            anchor_week = int(sub.iloc[best_idx]['week']) if 'week' in sub.columns else None
            anchor_x = best_x.copy()
            anchor_y = best_after

        # Week-to-week visual/status rule: green if latest output is higher than prior week; red if lower.
        if np.isnan(prior_week_output):
            improved = 'Baseline'
        elif latest_y > prior_week_output:
            improved = 'GREEN - higher than prior week'
        elif latest_y < prior_week_output:
            improved = 'RED - lower than prior week'
        else:
            improved = 'Neutral - same as prior week'

        kernel = Matern(nu=2.5)
        gp = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            n_restarts_optimizer=8,
            random_state=4210 + i
        )
        gp.fit(X_updated, Y_updated)

        mode_key = strat['mode']

        if FINAL_QUERY_OVERRIDE.get(i) is not None:
            # Benchmark-informed final query point.
            # Applied only when the benchmark's observed result is stronger than our own reviewed anchor.
            final_query = np.array(FINAL_QUERY_OVERRIDE[i], dtype=float)
            mode_label = 'Benchmark-based stronger final entry'

        elif mode_key == 'manual' and FINAL_QUERY_OVERRIDE.get(i) is not None:
            final_query = np.array(FINAL_QUERY_OVERRIDE[i], dtype=float)
            mode_label = 'Manual targeted query'

        elif mode_key == 'grid_then_ei':
            final_query = np.array(grid_search_f1(X_updated, Y_updated, n_grid=5))
            mode_label = 'Grid scan then EI'

        elif mode_key == 'anchor_perturb':
            x1_range  = strat.get('x1_range', (0.01, 0.03))
            x1_target = strat.get('x1_target', 0.020)
            candidates = anchor_perturb_f7(
                best_x, n_dims,
                radius=strat['perturb'],
                x1_range=x1_range,
                x1_target=x1_target
            )
            mean, std = gp.predict(candidates, return_std=True)
            scores = mean + strat['beta'] * std
            final_query = candidates[np.argmax(scores)]
            final_query[0] = float(np.clip(final_query[0], x1_range[0], x1_range[1]))
            mode_label = f"Anchor+perturb (beta={strat['beta']}, radius={strat['perturb']})"

        elif mode_key in ['exploit', 'anchored_exploit']:
            # Anchored trust-region Bayesian step.
            # Candidate selection remains GP/acquisition-based; the anchor only defines the search region.
            radius = strat['perturb']
            n_local = 32000
            n_trend = 5000

            eps = 1e-6
            search_anchor = anchor_x if mode_key == 'anchored_exploit' else best_x
            local_candidates = search_anchor + rng.uniform(-radius, radius, (n_local, n_dims))

            trend_candidates = []
            if strat.get('trend_follow', False) and len(X_updated) >= 2:
                # Trend-follow only where explicitly requested, mainly F5.
                prev_x = X_updated[-2]
                direction = search_anchor - prev_x
                trend_base = search_anchor + rng.uniform(0.00, 0.60, (n_trend, 1)) * direction
                trend_noise = rng.uniform(-radius/2, radius/2, (n_trend, n_dims))
                trend_candidates = trend_base + trend_noise

            if isinstance(trend_candidates, list) and len(trend_candidates) == 0:
                candidates = local_candidates
            else:
                candidates = np.vstack([local_candidates, trend_candidates])

            candidates = np.clip(candidates, eps, 1 - eps)

            mean, std = gp.predict(candidates, return_std=True)
            xi = strat.get('xi', 0.001)

            # Conservative EI/UCB hybrid: prioritize predicted improvement, but allow tiny uncertainty-driven moves.
            improvement = mean - best_after - xi
            ei_scores = np.where(
                improvement > 0,
                improvement + strat['beta'] * std,
                strat['beta'] * std * 0.01
            )
            final_query = candidates[np.argmax(ei_scores)]
            mode_label = f"Anchored trust-region BO around W{anchor_week} basin (radius={radius}, beta={strat['beta']}, xi={xi})"

        else:
            # Global UCB fallback.
            candidates = rng.uniform(0, 1, (30000, n_dims))
            mean, std = gp.predict(candidates, return_std=True)
            ucb = mean + strat['beta'] * std
            final_query = candidates[np.argmax(ucb)]
            mode_label = f"Global UCB (beta={strat['beta']})"

        final_query = validate_query(i, final_query, n_dims)
        query_str = format_query(final_query)

        original_reason = ORIGINAL_REASONING[i]
        if WEEK_MODE == 'observe':
            strategy_changed = 'No - observe mode'
            strategy_reason  = f"Original: {original_reason}"
        else:
            default_mode = DEFAULT_STRATEGY[i]['mode']
            if mode_key != default_mode:
                strategy_changed = 'YES - strategy adjusted'
            else:
                strategy_changed = 'No - kept same mode'
            strategy_reason = f"Function description: {original_reason} | latest-iteration final logic: {strat.get('reason', '')}"

        print(f'\nFunction {i} - {strat["note"]}')
        print(f'  Mode       : {mode_label}')
        print(f'  Points     : {len(Y_updated)} total')
        print(f'  Latest W{LATEST_WEEK}: {latest_y:.10g}')
        if np.isnan(prior_week_output):
            print(f'  Baseline   : Week 1 is neutral  [{improved}]')
        else:
            print(f'  Prior week : {prior_week_output:.10g} -> Latest {latest_y:.10g}  [{improved}]')
        print(f'  Best       : {best_before_latest:.10g} -> {best_after:.10g}')
        print(f'  Anchor     : Week {anchor_week} | y={anchor_y:.10g} | x={format_query(anchor_x)}')
        print(f'  Final Query : {query_str}')

        comparison_rows.append({
            'Function': fn_label,
            'Dims': n_dims,
            'Latest Week': LATEST_WEEK,
            'Points Available': len(Y_updated),
            'Prior Week Output': round(prior_week_output, 10) if not np.isnan(prior_week_output) else np.nan,
            'Best Before Latest': round(best_before_latest, 10),
            'Latest Output': round(latest_y, 10),
            'Best After Latest': round(best_after, 10),
            'Official Direction': 'MAXIMISE',
            'Colour Logic': 'Week 1 baseline neutral; from Week 2 onward GREEN=higher than prior week, RED=lower than prior week; GOLD STAR=highest value available',
            'Improved?': improved,
            'Function Description': original_reason,
            'Anchor Week': anchor_week,
            'Anchor Output': round(anchor_y, 10),
            'Anchor X': format_query(anchor_x),
            'Strategy Used': mode_label,
            'Strategy Changed?': strategy_changed,
            'Strategy Explanation': strategy_reason
        })

        new_query_rows.append({
            'Function': fn_label,
            'Dims': n_dims,
            'Best Output So Far': round(best_after, 10),
            'Function Description': original_reason,
            'Anchor Week': anchor_week,
            'Anchor Output': round(anchor_y, 10),
            'Anchor X': format_query(anchor_x),
            'Strategy Used': mode_label,
            'Strategy Changed?': strategy_changed,
            'Strategy Explanation': strategy_reason,
            'Final Query (formatted)': query_str
        })

    df_comparison = pd.DataFrame(comparison_rows)
    df_queries    = pd.DataFrame(new_query_rows)

    comparison_file = os.path.join(OUTPUT_PATH, f'comparison_table_w{LATEST_WEEK}.csv')
    monitor_file    = os.path.join(OUTPUT_PATH, f'best_value_monitor_w{LATEST_WEEK}.csv')
    queries_file    = os.path.join(OUTPUT_PATH, f'final_queries_week{LATEST_WEEK}.csv')

    df_comparison.to_csv(comparison_file, index=False)
    df_queries.to_csv(queries_file, index=False)
    if 'df_best_monitor' in globals():
        df_best_monitor.to_csv(monitor_file, index=False)

    print('\n' + '='*70)
    print(f'WEEK {LATEST_WEEK} FINAL QUERY RECORDS')
    print('='*70)
    for _, row in df_queries.iterrows():
        print(f"  {row['Function']}: {row['Final Query (formatted)']}")

    print(f'\nCSVs saved to: {OUTPUT_PATH}')
    print(f'Comparison: {comparison_file}')
    print(f'Queries   : {queries_file}')
    if 'monitor_file' in locals():
        print(f'Monitor   : {monitor_file}')





pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

try:
    display(df_comparison[['Function','Official Direction','Colour Logic','Prior Week Output','Latest Output','Best After Latest','Improved?','Strategy Used']])
except NameError:
    print('Run Cell 6 first to generate df_comparison.')
except KeyError as e:
    print(f'Comparison display skipped because columns changed: {e}')

try:
    display(df_best_monitor)
except NameError:
    print('Run Cell 2A first to generate df_best_monitor.')

try:
    week12_logic_cols = [
        'Function', 'Latest Week Available', 'Latest Output',
        'Final-week Anchor Week', 'Final-week Anchor Output',
        'Final-week Anchor Logic', 'Anchor X'
    ]
    display(df_best_monitor[week12_logic_cols])
except NameError:
    print('Run Cell 2A first to display the Final-week best-week anchor logic table.')
except KeyError as e:
    print(f'Final-week logic table skipped because columns changed: {e}')

## 4. Strategy metadata:
Defines the strategy notes used in tables.

In [ ]:
# STRATEGY_EVOLUTION_TABLE
# This table summarizes the actual strategy applied through Final-week final results.

strategy_evolution = pd.DataFrame({
    'Function': ['F1','F2','F3','F4','F5','F6','F7','F8'],
    'Function Description': [FUNCTION_DESCRIPTIONS[i] for i in range(1, 9)],
    'Week 1': ['Initial probe','Initial probe','Initial probe','Initial probe','Initial probe','Initial probe','Initial probe','Initial probe'],
    'Week 2': ['Broad exploration','Exploration','Exploration','Aggressive exploration','Exploitation after signal','Moderate exploration','Exploration','Exploration'],
    'Week 3': ['Grid-style exploration','UCB / exploratory query','Local exploitation','Continued exploration','Local exploitation','UCB exploration','Anchor + perturb','Local exploitation'],
    'Week 4': ['Grid / exploratory probing','Local exploitation','Tight local exploitation','Recovery attempt','Tight exploitation','Local exploitation','Anchor + perturb','Local exploitation'],
    'Week 5': ['Corner exploration','Manual local exploitation','Manual refinement','Manual repositioning','Manual tight exploitation','Trust-region exploitation','Manual local perturbation','Manual structured exploitation'],
    'Week 6': ['Edge/corner exploration','Rollback toward best basin','Rollback to best basin','Exploit recovered basin','Micro-tuning around peak','Trust-region refinement','Rollback + tiny perturbation','Stabilization near best basin'],
    'Week 7': ['Aggressive exploration','Rollback + refine','Local refinement','Controlled exploitation','Tight exploitation','Trust-region refinement','Very tight exploitation','Stabilize + refine'],
    'Week 8': ['Central exploration','Local refinement','Local refinement','Controlled exploitation','Micro-tuning around peak','Rollback to Week 6 structure','Return to Week 6 plateau','Micro-refinement'],
    'Week 9': ['Follow-up near W8 signal','Returned near W7 basin','Repeated stable point','Revisited W3-like basin','New best / directional exploit','Returned near best basin','Boundary micro-tune','Plateau micro-refine'],
    'Week 10': ['Function-aware recovery','Function-aware local rollback','Local refinement around best basin','Controlled many-optima search','Strong yield-ridge exploit','Residual-minimisation refinement','Boundary-sensitive micro-tune','High-dimensional micro-refinement'],
    'Week 11 Actual Result': ['Regression vs historical best','Still below Week 7 peak','New/latest best basin','Below reviewed Week 5 basin','Strong late yield basin','Regression vs Week 10','New/latest best region','High-performing plateau'],
    'Final Strategy': ['Re-anchor to W8 and exploit locally','Return to W7 with moderate exploration','Exploit W11 with small perturbations','Recover W5 and fine-tune','Aggressively exploit W11 yield ridge','Hybrid recovery around W10','Exploit W11 region','Conservative exploitation around W11 plateau'],
})

strategy_evolution

## 5. Dashboard design and visual functions:
Defines the visual theme, chart helpers, cards, 3D model-belief plot, trend line, table rendering, and export functions.

In [ ]:
# ============================================================
# Dashboard Generation
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import textwrap
import base64
import io
import matplotlib.image as mpimg
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ----------------------------
# Visual metadata
# ----------------------------
FUNCTION_LABELS = {
    1: 'Radiation Field Detection',
    2: 'Noisy ML Log-Likelihood',
    3: 'Drug Compound Side Effects',
    4: 'Warehouse Placement',
    5: 'Chemical Process Yield',
    6: 'Recipe Quality Scoring',
    7: 'ML Hyperparameter Tuning',
    8: 'Neural Network Configuration',
}

FUNCTION_DIMS = {i: len(WEEKLY_RESULTS[max(WEEKLY_RESULTS.keys())][i]['x']) for i in range(1, 9)}
LATEST_WEEK = max(WEEKLY_RESULTS.keys())

# ----------------------------
# Output path
# ----------------------------
VISUAL_OUTPUT_PATH = os.path.join(OUTPUT_PATH, f"BBO_outputs_W{LATEST_WEEK}")
os.makedirs(VISUAL_OUTPUT_PATH, exist_ok=True)

# ----------------------------
# Design system
# ----------------------------
BG_PAGE = '#050B14'
BG_PANEL = '#071424'
BG_CARD = '#081A2B'
BG_CARD2 = '#0B2138'
BORDER = '#1B4B73'
BORDER_SOFT = '#163B5A'
CYAN = '#00D7FF'
BLUE = '#168BFF'
GREEN = '#18D879'
RED = '#FF3048'
WHITE = '#FFFFFF'
GRID = '#17314A'
PURPLE = '#D3145A'
PURPLE_MID = PURPLE
GOLD = PURPLE   # reviewed best week / anchor marker

# Dashboard colour rule requested:
# BLUE = Week 1 baseline and should not be compared/changed.
# From Week 2 onward: GREEN = current value is higher than the immediately prior week;
# RED = current value is lower than the immediately prior week.
# If two consecutive weeks have the same value, the current week inherits the prior week's colour.

plt.rcParams.update({
    'figure.facecolor': BG_PAGE,
    'axes.facecolor': BG_PANEL,
    'savefig.facecolor': BG_PAGE,
    'font.family': 'DejaVu Sans',
    'text.color': WHITE,
    'axes.labelcolor': WHITE,
    'xtick.color': WHITE,
    'ytick.color': WHITE,
    'axes.edgecolor': BORDER,
})

# ----------------------------
# Embedded logo
# ----------------------------
LOGO_BASE64 = '''...'''


def remove_black_background(img, threshold=0.08, softness=0.10):
    """Turn near-black logo background pixels transparent while preserving the logo."""
    if img is None:
        return None
    arr = np.array(img).astype(float)
    if arr.max() > 1:
        arr = arr / 255.0
    if arr.shape[-1] == 3:
        alpha = np.ones(arr.shape[:2], dtype=float)
        arr = np.dstack([arr, alpha])
    rgb = arr[..., :3]
    brightness = rgb.max(axis=2)
    alpha = np.clip((brightness - threshold) / softness, 0, 1)
    alpha[brightness > (threshold + softness)] = 1
    arr[..., 3] = arr[..., 3] * alpha
    return arr


def load_logo():
    raw = None
    try:
        if LOGO_BASE64.strip():
            raw = mpimg.imread(io.BytesIO(base64.b64decode(LOGO_BASE64)))
    except Exception:
        raw = None
    if raw is None:
        for p in ['BBO2(1).png', '/mnt/data/BBO2(1).png', 'BBO2.png', '/mnt/data/BBO2.png']:
            try:
                if os.path.exists(p):
                    raw = mpimg.imread(p)
                    break
            except Exception:
                pass
    return remove_black_background(raw) if raw is not None else None

LOGO_IMG = load_logo()


# ----------------------------
# Safe save helper
# ----------------------------
def safe_savefig(fig, path, dpi=160):
    """
    Safer Windows save:
    - cleans invalid filename characters
    - tries requested path first
    - if Windows rejects it, saves to C:\\temp\\BBO_outputs_safe
    """
    import os
    import re

    requested_path = path

    try:
        folder, filename = os.path.split(path)
        os.makedirs(folder, exist_ok=True)

        # Clean only the filename, not the drive/path separators
        filename = re.sub(r'[<>:"/\\\\|?*]', '_', filename)
        safe_path = os.path.normpath(os.path.join(folder, filename))

        fig.savefig(
            safe_path,
            dpi=dpi,
            facecolor=BG_PAGE,
            bbox_inches=None,
            pad_inches=0.05
        )
        plt.close(fig)
        print(f"Saved -> {safe_path}")

    except OSError:
        fallback_folder = r"C:\\temp\\BBO_outputs_safe"
        os.makedirs(fallback_folder, exist_ok=True)

        fallback_name = os.path.basename(requested_path)
        fallback_name = re.sub(r'[<>:"/\\\\|?*]', '_', fallback_name)
        fallback_path = os.path.normpath(os.path.join(fallback_folder, fallback_name))

        fig.savefig(
            fallback_path,
            dpi=dpi,
            facecolor=BG_PAGE,
            bbox_inches=None,
            pad_inches=0.05
        )
        plt.close(fig)
        print(f"Original path failed: {requested_path}")
        print(f"Saved instead to -> {fallback_path}")

# ----------------------------
# Helpers
# ----------------------------
def get_fn_inputs_outputs(fn):
    weeks = sorted(WEEKLY_RESULTS.keys())
    xs = [WEEKLY_RESULTS[w][fn]['x'] for w in weeks]
    ys = [WEEKLY_RESULTS[w][fn]['y'] for w in weeks]
    return weeks, np.array(xs, dtype=float), np.array(ys, dtype=float)

def sci(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '—'
    if abs(v) >= 1000: return f'{v:,.1f}'
    if abs(v) >= 10: return f'{v:.2f}'
    if abs(v) >= 1: return f'{v:.3f}'
    if abs(v) >= 1e-3: return f'{v:.4f}'
    return f'{v:.2e}'

def change_label(prev, curr):
    if prev is None:
        return '—'
    return sci(curr - prev)

def highest_week_index(ys, fn=None):
    """Return reviewed best-week index for dashboard text.
    Uses REVIEWED_BEST_WEEK_FOR_CHART when available so the text matches the strategy anchors.
    Falls back to the numeric maximum if no reviewed anchor exists.
    """
    if fn is not None and 'REVIEWED_BEST_WEEK_FOR_CHART' in globals():
        wk = REVIEWED_BEST_WEEK_FOR_CHART.get(int(fn))
        if wk is not None:
            return max(0, min(int(wk) - 1, len(ys) - 1))
    return int(np.argmax(np.asarray(ys, dtype=float)))

def status_from_series(ys):
    """Latest week status using raw value comparison against the prior week."""
    ys = np.asarray(ys, dtype=float)
    if len(ys) < 2:
        return 'BASELINE'
    if ys[-1] > ys[-2]:
        return 'IMPROVED'
    if ys[-1] < ys[-2]:
        return 'DOWNTREND'
    return 'FLAT'

def status_color(status):
    if status == 'IMPROVED':
        return GREEN
    if status == 'DOWNTREND':
        return RED
    if status == 'BASELINE':
        return BLUE
    return WHITE

def week_color(ys, idx, mark_best=True):
    """
    Weekly colour logic:
    - W1 is always baseline blue.
    - W2+ is green when the raw value is higher than the prior week.
    - W2+ is red when the raw value is lower than the prior week.
    - Equal consecutive values inherit the previous week's colour.
    """
    ys = np.asarray(ys, dtype=float)
    if idx == 0:
        return BLUE
    if ys[idx] > ys[idx-1]:
        return GREEN
    if ys[idx] < ys[idx-1]:
        return RED
    return week_color(ys, idx-1, mark_best=mark_best)

def week_marker(ys, idx):
    return 'o'

def week_edge_color(ys, idx):
    return WHITE

def week_marker_size(ys, idx, base=135):
    return base

def add_colour_legend(fig, x=0.655, y=0.860, fontsize=18):
    """legend box"""
    items = [
        (BLUE, 'Baseline (W1)'),
        (GREEN, 'Higher'),
        (RED, 'Lower'),
    ]
    x0 = x
    y0 = y
    steps = [0.085, 0.067, 0.000]
    for i, (col, label) in enumerate(items):
        fig.add_artist(plt.Line2D([x0, x0+0.011], [y0, y0], transform=fig.transFigure,
                                  color=col, lw=6, solid_capstyle='round', zorder=21))
        fig.text(x0+0.015, y0-0.004, label, fontsize=fontsize, color=WHITE, va='center', ha='left', zorder=21)
        x0 += steps[i]

def safe_log_height(values):
    vals = np.asarray(values, dtype=float)
    if len(vals) == 0:
        return vals

    finite = vals[np.isfinite(vals)]
    if len(finite) == 0:
        return np.ones_like(vals) * 0.55

    use_log = np.all(finite > 0) and (np.nanmax(finite) / max(np.nanmin(finite), 1e-300) > 1e4)
    raw = np.log10(np.abs(vals) + 1e-300) if use_log else vals.copy()

    mn, mx = np.nanmin(raw), np.nanmax(raw)
    if abs(mx - mn) < 1e-12:
        return np.ones_like(raw) * 0.70
    h = (raw - mn) / (mx - mn)
    return 0.24 + 0.92 * h

def draw_panel(fig, x, y, w, h, edge=BORDER, face=BG_PANEL, lw=1.3, alpha=1.0, radius=0.01, z=0):
    patch = mpatches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f'round,pad=0.004,rounding_size={radius}',
        transform=fig.transFigure,
        facecolor=face,
        edgecolor=edge,
        linewidth=lw,
        alpha=alpha,
        zorder=z,
    )
    fig.patches.append(patch)
    return patch


def _tight_crop_logo(img, threshold=10):
    """Remove transparent/dark padding around the logo."""
    if img is None:
        return img

    arr = np.asarray(img)

    if arr.ndim == 3 and arr.shape[-1] == 4:
        mask = arr[:, :, 3] > threshold
    else:
        gray = arr[:, :, :3].mean(axis=2)
        mask = gray > 18

    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]

    if len(rows) == 0 or len(cols) == 0:
        return arr

    return arr[rows.min():rows.max()+1, cols.min():cols.max()+1]


def add_fixed_logo(fig, x=0.020, y=0.910, w=0.130, h=0.070):
    """
    Fixed logo after the red line.
    Keeps proportions, avoids horizontal stretching, and fills the logo box neatly.
    """
    if LOGO_IMG is None:
        ax = fig.add_axes([x, y, w, h], zorder=20)
        ax.axis("off")
        ax.patch.set_alpha(0)
        ax.text(
            0.5, 0.5, "black\nbox",
            ha="center", va="center",
            fontsize=22, fontweight="bold", color=WHITE
        )
        return ax

    img = _tight_crop_logo(LOGO_IMG)

    img_h, img_w = img.shape[:2]
    img_ratio = img_w / img_h
    box_ratio = w / h

    # Fit proportionally inside the box
    if img_ratio > box_ratio:
        final_w = w
        final_h = w / img_ratio
    else:
        final_h = h
        final_w = h * img_ratio

    # Center inside the target logo box
    final_x = x + (w - final_w) / 2
    final_y = y + (h - final_h) / 2

    ax = fig.add_axes([final_x, final_y, final_w, final_h], zorder=20)
    ax.axis("off")
    ax.patch.set_alpha(0)
    ax.imshow(img)

    return ax


add_logo = add_fixed_logo


def add_header(fig, fn=None, title=None, subtitle=None):
    """
    Unified non-overlapping header.
    """
    HEADER_X = 0.010
    HEADER_Y = 0.905
    HEADER_W = 0.980
    HEADER_H = 0.080

    RED_X = 0.012
    RED_Y = 0.908
    RED_W = 0.007
    RED_H = 0.074

    LOGO_X = RED_X + RED_W - 0.020
    LOGO_Y = 0.820
    LOGO_W = 0.200
    LOGO_H = 0.250

    TITLE_X = LOGO_X + LOGO_W - 0.020
    SUBTITLE_X = TITLE_X

    draw_panel(fig, HEADER_X, HEADER_Y, HEADER_W, HEADER_H,
               edge=BORDER, face=BG_CARD, lw=1.2, z=1)

    fig.patches.append(
        mpatches.Rectangle(
            (RED_X, RED_Y), RED_W, RED_H,
            transform=fig.transFigure,
            facecolor=RED,
            edgecolor="none",
            zorder=3
        )
    )

    add_fixed_logo(fig, x=LOGO_X, y=LOGO_Y, w=LOGO_W, h=LOGO_H)

    if fn is not None:
        fig.text(TITLE_X, 0.955, f"F{fn:02d}",
                 fontsize=37, fontweight="bold", color=CYAN, va="center")
        fig.text(TITLE_X + 0.075, 0.957,
                 (title or FUNCTION_LABELS[fn]).upper(),
                 fontsize=37, fontweight="bold", color=WHITE, va="center")
    else:
        fig.text(TITLE_X, 0.955, "BLACK-BOX OPTIMISATION",
                 fontsize=45, fontweight="bold", color=WHITE, va="center")

    if subtitle:
        # Sit roughly midway between the title row (0.955) and the
        # header box's bottom edge (HEADER_Y=0.905)
        fig.text(SUBTITLE_X, 0.925, subtitle,
                 fontsize=22, color=WHITE, va="center")

# ----------------------------
# 3D visuals
# ----------------------------

def draw_3d_bars(ax, ys, weeks=None, title=None, compact=False, fn=None):
    """
    Clean stable 3D weekly bars for overview cards.
    """
    ys = np.asarray(ys, dtype=float)
    weeks = weeks or list(range(1, len(ys)+1))
    heights = safe_log_height(ys)

    ax.set_facecolor(BG_PANEL)

    dx = 0.58 if compact else 0.66
    dy = 0.46 if compact else 0.56
    spacing = 1.16 if compact else 1.24

    x_centers = [i * spacing + dx / 2 for i in range(len(ys))]

    for i in reversed(range(len(ys))):
        wk, yv, hh = weeks[i], ys[i], heights[i]
        col = week_color(ys, i)
        x0 = i * spacing
        y0 = 0.0
        x_center = x_centers[i]

        ax.bar3d(
            x0, y0, 0,
            dx, dy, hh,
            color=col,
            shade=True,
            alpha=0.92,
            edgecolor=col,
            linewidth=0.7
        )
    # Bounds
    x_min = -0.08
    x_max = (len(ys)-1) * spacing + dx + 0.08
    z_max = max(1.0, float(np.max(heights)))

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-0.08, 0.82)
    ax.set_zlim(0, z_max * 1.15)

    # X labels: 
    ax.set_xticks(x_centers)
    ax.set_xticklabels(
        [f'W{w}' for w in weeks],
        fontsize=9 if compact else 12,
        color=WHITE,
        fontweight='bold'
    )

    # Hide y labels, keep z labels for reference
    ax.set_yticks([])

    z_ticks = np.linspace(0, z_max, 5)
    ax.set_zticks(z_ticks)
    ax.set_zticklabels(
        [f'{z:.1f}' for z in z_ticks],
        fontsize=8 if compact else 10,
        color=(1, 1, 1, 0.68)
    )

    # Remove axis titles
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_zlabel('')

    ax.tick_params(axis='x', colors=WHITE, pad=0)
    ax.tick_params(axis='z', colors=WHITE, pad=0)

    ax.view_init(elev=20, azim=-50)
    ax.set_box_aspect((4.6, 0.85, 0.82))

    try:
        ax.dist = 5.2
    except Exception:
        pass

    # Soft gridlines:
    # show z-grid/horizontal reference lines; suppress x/y grid clutter.
    ax.grid(True)
    ax.xaxis._axinfo["grid"]["linewidth"] = 0.0
    ax.yaxis._axinfo["grid"]["linewidth"] = 0.0
    ax.zaxis._axinfo["grid"]["color"] = (0.22, 0.50, 0.72, 0.32)
    ax.zaxis._axinfo["grid"]["linewidth"] = 0.75
    ax.zaxis._axinfo["grid"]["linestyle"] = "-"

    # Pane styling
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.set_facecolor((0.03, 0.08, 0.14, 1.0))
        pane.set_edgecolor((0.09, 0.20, 0.30, 0.35))

    if title:
        ax.set_title(title, fontsize=18, color=WHITE, fontweight='bold', pad=8)

def projected_xy(xs):
    xs = np.asarray(xs, dtype=float)
    x1 = xs[:, 0]
    x2 = xs[:, 1] if xs.shape[1] >= 2 else np.zeros_like(x1)
    return x1, x2

def surface_from_points(xs, ys, n=110):
    xgrid = np.linspace(0, 1, n)
    ygrid = np.linspace(0, 1, n)
    X, Y = np.meshgrid(xgrid, ygrid)
    qx, qy = projected_xy(xs)
    log_vals = np.log10(np.abs(ys) + 1e-300)
    mn, mx = np.min(log_vals), np.max(log_vals)
    norm = np.zeros_like(log_vals) + 0.5 if abs(mx - mn) < 1e-12 else (log_vals - mn) / (mx - mn)
    Z = np.zeros_like(X) + 0.05
    for ox, oy, amp in zip(qx, qy, norm):
        sigma = 0.055 + 0.06 * (1 - amp)
        Z += (0.20 + 0.95 * amp) * np.exp(-((X-ox)**2 + (Y-oy)**2) / (2*sigma**2))
    Z += 0.06 * np.sin(3*np.pi*X) * np.cos(2*np.pi*Y)
    Z = Z - Z.min()
    Z = Z / (Z.max() + 1e-12)
    return X, Y, Z


def draw_3d_mountain(ax, xs, ys):
    """Render the model-belief surface large inside its existing panel."""
    ax.set_facecolor(BG_PANEL)
    ax.margins(0)

    X, Y, Z = surface_from_points(xs, ys, n=170)
    cmap = LinearSegmentedColormap.from_list(
        'belief',
        ['#003B8E', '#0068C9', '#00A3FF', '#00D97A', '#FFE25A', '#FF7A00', '#FF2424']
    )

    ax.plot_surface(
        X, Y, Z,
        cmap=cmap,
        linewidth=0,
        antialiased=True,
        alpha=0.98,
        rstride=1,
        cstride=1
    )
    ax.contour(
        X, Y, Z,
        zdir='z',
        offset=-0.08,
        levels=10,
        cmap=cmap,
        linewidths=1.0,
        alpha=0.8
    )

    qx, qy = projected_xy(xs)
    zq = []
    for a, b in zip(qx, qy):
        ix = np.argmin(np.abs(X[0] - a))
        iy = np.argmin(np.abs(Y[:, 0] - b))
        zq.append(Z[iy, ix] + 0.035)
    zq = np.asarray(zq)

    ax.plot(qx, qy, zq, color=WHITE, linestyle='--', linewidth=3.2, alpha=0.97)
    for i, (a, b, c) in enumerate(zip(qx, qy, zq), start=1):
        col = week_color(ys, i - 1)
        ax.scatter(
            [a], [b], [c],
            color=col,
            s=week_marker_size(ys, i - 1, base=135),
            marker=week_marker(ys, i - 1),
            edgecolor=week_edge_color(ys, i - 1),
            linewidth=1.6,
            depthshade=False
        )
        # Label only the key route markers to keep the larger plot clean.
        show_label = (i == 1) or (i == len(qx)) or (i == int(np.nanargmax(np.asarray(ys, dtype=float))) + 1) or (i in [2, 3])
        if show_label:
            ax.text(
                a, b, c + 0.070,
                f'W{i}',
                color=WHITE,
                fontsize=19,
                fontweight='bold',
                ha='center',
                va='bottom',
                bbox=dict(boxstyle='round,pad=0.22', fc=BG_CARD2, ec=col, lw=1.2, alpha=0.95)
            )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_zlim(-0.08, 1.10)

    ax.set_xlabel('Input dimension 1', labelpad=8, fontsize=16, color=WHITE)
    ax.set_ylabel('Input dimension 2', labelpad=8, fontsize=16, color=WHITE)
    ax.set_zlabel('Projected mean', labelpad=7, fontsize=16, color=WHITE)
    ax.tick_params(axis='both', which='major', labelsize=13, colors=WHITE, pad=0)
    ax.view_init(elev=25, azim=-50)
    
    try:
        ax.set_box_aspect((2.85, 1.55, 0.75), zoom=1.45)
    except TypeError:
        ax.set_box_aspect((2.55, 1.45, 0.72))
    try:
        ax.dist = 2.15
    except Exception:
        pass

    ax.grid(True, color=GRID, alpha=0.45)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.set_facecolor((0.03, 0.08, 0.14, 1.0))
        pane.set_edgecolor(BORDER)


def draw_trend(ax, weeks, ys, title='OUTPUT TREND', fn=None):
    """Draw the output trajectory with non-overlapping axis labels and peak annotations."""
    ax.set_facecolor(BG_PANEL)
    vals = np.log10(np.abs(ys) + 1e-300)

    ax.plot(
        weeks,
        vals,
        color=WHITE,
        linestyle='--',
        linewidth=2.4,
        marker='o',
        markersize=8,
        markerfacecolor=BG_PAGE,
        markeredgecolor=WHITE,
        zorder=2
    )

    span = float(np.nanmax(vals) - np.nanmin(vals))
    pad = max(0.75, span * 0.32)
    ymin = float(np.nanmin(vals) - pad * 0.70)
    ymax = float(np.nanmax(vals) + pad * 0.95)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(min(weeks) - 0.95, max(weeks) + 1.35)

    best_idx = int(np.nanargmax(np.asarray(ys, dtype=float)))
    for idx, (w, raw, lv) in enumerate(zip(weeks, ys, vals)):
        col = week_color(ys, idx)
        ax.scatter(
            [w],
            [lv],
            s=week_marker_size(ys, idx, base=135),
            marker=week_marker(ys, idx),
            color=col,
            edgecolor=week_edge_color(ys, idx),
            linewidth=1.4,
            zorder=4
        )

        if idx == 0:
            ax.annotate(
                f'{sci(raw)} (Baseline)',
                xy=(w, lv),
                xytext=(-8, -34),
                textcoords='offset points',
                color=col,
                fontsize=15,
                fontweight='bold',
                ha='left',
                va='top',
                clip_on=False,
                zorder=6
            )
        elif idx == best_idx:
            offset_x = -6 if w >= max(weeks) - 1 else 8
            ha = 'right' if w >= max(weeks) - 1 else 'left'
            ax.annotate(
                f'{sci(raw)} (Best)',
                xy=(w, lv),
                xytext=(offset_x, 20),
                textcoords='offset points',
                color=GREEN,
                fontsize=15,
                fontweight='bold',
                ha=ha,
                va='bottom',
                clip_on=False,
                zorder=6
            )

    ax.set_xticks(weeks)
    ax.set_xticklabels([f'W{w}' for w in weeks], fontsize=15, color=WHITE)
    ax.set_xlabel('')
    ax.set_ylabel('log10(|output|)', fontsize=15, color=WHITE, labelpad=16)
    ax.tick_params(axis='x', pad=8)
    ax.tick_params(axis='y', pad=8)
    ax.grid(True, color=GRID, alpha=0.65)
    ax.set_title(title, fontsize=22, color=WHITE, fontweight='bold', loc='left', pad=14)
    for spine in ax.spines.values():
        spine.set_color(BORDER)

# ----------------------------
# Strategy text utilities
# ----------------------------
STRATEGY_TEXT = {
    1: {1:'Initial baseline probe (space-filling)', 2:'UCB broad exploration', 3:'Grid-style exploration', 4:'Grid / exploratory probing', 5:'Corner exploration', 6:'Edge/corner exploration', 7:'Aggressive exploration'},
    2: {1:'Initial baseline probe', 2:'UCB exploration', 3:'UCB / exploratory query', 4:'Local exploitation', 5:'Manual local exploitation', 6:'Rollback toward best basin', 7:'Rollback + refine'},
    3: {1:'Initial baseline probe', 2:'UCB exploration', 3:'Local exploitation', 4:'Tight local exploitation', 5:'Manual refinement', 6:'Rollback to best basin', 7:'Local refinement'},
    4: {1:'Initial baseline probe', 2:'UCB aggressive exploration', 3:'UCB continued exploration', 4:'UCB / recovery attempt', 5:'Manual repositioning', 6:'Exploit recovered basin', 7:'Controlled exploitation'},
    5: {1:'Initial baseline probe', 2:'UCB exploitation', 3:'Local exploitation', 4:'Tight exploitation', 5:'Manual tight exploitation', 6:'Micro-tuning around peak', 7:'Tight exploitation'},
    6: {1:'Initial baseline probe', 2:'UCB moderate exploration', 3:'UCB exploration', 4:'Local exploitation', 5:'Trust-region exploitation', 6:'Trust-region refinement', 7:'Trust-region refinement'},
    7: {1:'Initial baseline probe', 2:'UCB exploration', 3:'Anchor + perturb', 4:'Anchor + perturb', 5:'Manual local perturbation', 6:'Rollback + tiny perturbation', 7:'Very tight exploitation'},
    8: {1:'Initial baseline probe', 2:'UCB exploration', 3:'Local exploitation', 4:'Local exploitation', 5:'Manual structured exploitation', 6:'Stabilization near best basin', 7:'Stabilize + refine'},
}
OBS_TEXT = {
    1: {1:'Baseline established', 2:'No stable signal yet', 3:'Active region still unclear', 4:'Signal remains sparse', 5:'New area tested', 6:'Still no meaningful signal', 7:'Still no active region'},
    2: {1:'Baseline established', 2:'Early good region found', 3:'Query moved away', 4:'Best basin identified', 5:'Moved away from peak', 6:'Partial recovery', 7:'Best value so far'},
    3: {1:'Baseline established', 2:'Worse region identified', 3:'Reviewed best basin observed', 4:'Moved away from reviewed best', 5:'Overshot the basin', 6:'Recovered close to good region', 7:'Slight improvement'},
    4: {1:'Baseline established', 2:'Poor local region', 3:'Partial recovery', 4:'Still unstable', 5:'Recovery started', 6:'Improving basin confirmed', 7:'Continued improvement'},
    5: {1:'Baseline established', 2:'Strong high-output area found', 3:'Still near strong basin', 4:'Improvement continued', 5:'New peak identified', 6:'Peak strengthened', 7:'Best peak so far'},
    6: {1:'Baseline established', 2:'Moderate improvement', 3:'Reviewed best basin observed', 4:'Moved away from reviewed best', 5:'Small gain', 6:'Recovered but not anchor', 7:'Moved too far'},
    7: {1:'Baseline established', 2:'Strong response detected', 3:'Plateau identified', 4:'Plateau maintained', 5:'Moved too far', 6:'Best plateau recovered', 7:'Slight drop on plateau'},
    8: {1:'Baseline established', 2:'Small gain', 3:'Good structure found', 4:'Best structure observed', 5:'Slightly worse', 6:'Stabilized near best', 7:'Stable high value'},
}
FINAL_QUERY_NOTE = {
    1: ('Central exploration in a new region', 'No meaningful signal yet. Test the centre after edge/corner attempts.'),
    2: ('Very local refinement around Final week basin', 'Final week produced the best value so far; keep changes minimal.'),
    3: ('Return near the reviewed best basin', 'Week 3 is the reviewed champion for Function 3; refine close to that area using maximisation colouring.'),
    4: ('Controlled exploitation of reviewed Week 4 basin', 'Week 4 is the reviewed champion for Function 4; continue small controlled movement around it.'),
    5: ('Micro-tuning around the peak', 'The peak continues improving; stay very close to Final week.'),
    6: ('Rollback to Week 6 structure', 'Final week dropped; return to Week 6 and shrink the step.'),
    7: ('Return to Week 6 plateau', 'Final week slightly dropped; keep the Week 6 plateau structure.'),
    8: ('Micro-refinement near best-known structure', 'Performance is stable and high; avoid large movements.'),
}

def get_final_query(fn):
    return WEEKLY_RESULTS[LATEST_WEEK][fn]['x']

# ----------------------------
# Detailed function dashboard
# ----------------------------

def draw_week_card(fig, fn, wk, x, y, w, h, ys):
    """Compact one-row week card: readable text, no overlap, fill color matches border."""
    idx = wk - 1
    col = week_color(ys, idx)
    draw_panel(fig, x, y, w, h, edge=col, face=col, lw=1.8, alpha=0.16, radius=0.006, z=1)
    pad_x = 0.002 * w
    top = y + h
    fig.text(x+pad_x, top-0.040, f'WEEK {wk}', fontsize=22, color=col, fontweight='bold')
    fig.text(x+pad_x, top-0.074, 'OUTPUT', fontsize=14.5, color=WHITE, fontweight='bold')
    fig.text(x+pad_x, top-0.108, sci(ys[idx]), fontsize=22, color=WHITE, fontweight='bold')
    fig.lines.append(plt.Line2D([x+pad_x, x+w-pad_x], [top-0.128, top-0.128], transform=fig.transFigure, color=col, lw=1.1, alpha=0.85))
    fig.text(x+pad_x, top-0.154, 'OBSERVATION', fontsize=14.5, color=WHITE, fontweight='bold')
    obs = OBS_TEXT.get(fn, {}).get(wk, 'Updated search understanding')
    fig.text(x+pad_x, top-0.183, '\n'.join(textwrap.wrap(obs.replace('\n',' '), width=20)), fontsize=17, color=WHITE, va='top', linespacing=1.12)
    fig.lines.append(plt.Line2D([x+pad_x, x+w-pad_x], [y+0.055, y+0.055], transform=fig.transFigure, color=col, lw=1.0, alpha=0.75))
    fig.text(x+pad_x, y+0.034, 'STRATEGY APPLIED', fontsize=16, color=WHITE, fontweight='bold')
    strat = STRATEGY_TEXT.get(fn, {}).get(wk, 'Adaptive Bayesian strategy')
    fig.text(x+pad_x, y+0.010, '\n'.join(textwrap.wrap(strat.replace('\n',' '), width=20)), fontsize=17, color=WHITE, va='bottom', linespacing=1.10)


def plot_function_dashboard(fn):
    LEFT_BOUND = 0.012
    RIGHT_BOUND = 0.988
    TOTAL_WIDTH = RIGHT_BOUND - LEFT_BOUND
    weeks, xs, ys = get_fn_inputs_outputs(fn)
    best_idx = int(np.argmax(ys))
    status = status_from_series(ys)
    s_col = status_color(status)
    label = FUNCTION_LABELS[fn]
    dims = FUNCTION_DIMS[fn]

    fig = plt.figure(figsize=(42, 24), facecolor=BG_PAGE)
    subtitle = f'{dims}D INPUT SPACE    |    {LATEST_WEEK}-WEEK OPTIMISATION    |    PEAK: {sci(ys[best_idx])} (WEEK {weeks[best_idx]})    |    ADAPTIVE BAYESIAN STRATEGY'
    add_header(fig, fn=fn, title=label, subtitle=subtitle)

    # Top visual panels with gaps between boxes
    TOP_Y = 0.510
    TOP_H = 0.365

    # Performance Summary box
    LEFT_BOUND = 0.010
    RIGHT_BOUND = 0.990
    TOTAL_WIDTH = RIGHT_BOUND - LEFT_BOUND

    GAP = 0.014

    # Week-card grid used below
    CARD_LEFT = 0.012
    CARD_GAP_X = 0.018
    CARD_COLS = LATEST_WEEK
    CARD_W = (0.976 - (CARD_COLS - 1) * CARD_GAP_X) / CARD_COLS

    # Top panels
    LEFT_X = LEFT_BOUND
    LEFT_W = 0.345

    # 3D heat/map visual
    RIGHT_W = 0.160
    RIGHT_X = RIGHT_BOUND - RIGHT_W
    MID_X = LEFT_X + LEFT_W + GAP
    MID_W = RIGHT_X - MID_X - GAP

    # Left panel: trend
    draw_panel(fig, LEFT_X, TOP_Y, LEFT_W, TOP_H, edge=BORDER, face=BG_PANEL, lw=1.1, radius=0.006, z=0)
    ax_trend = fig.add_axes(
        [LEFT_X + 0.040, TOP_Y + 0.070, LEFT_W - 0.066, TOP_H - 0.125],
        zorder=2
    )
    draw_trend(ax_trend, weeks, ys, title='OUTPUT TRAJECTORY (LOG SCALE)', fn=fn)

    # Middle panel: 3D mountain/map
    draw_panel(fig, MID_X, TOP_Y, MID_W, TOP_H, edge=BORDER, face=BG_PANEL, lw=1.1, radius=0.006, z=0)
    ax_map = fig.add_axes(
        [MID_X + 0.020, TOP_Y + 0.045, MID_W - 0.040, TOP_H - 0.095],
        projection='3d',
        zorder=2
    )
    draw_3d_mountain(ax_map, xs, ys)
    fig.text(
    MID_X + MID_W/2,          # center of the panel
    TOP_Y + TOP_H - 0.025,    # INSIDE the panel
    'SEARCH PATH & MODEL BELIEF (3D)',
    ha='center',
    fontsize=25,
    color=WHITE,
    fontweight='bold'
)

    # Right panel: Performance Summary
    draw_panel(fig, RIGHT_X, TOP_Y, RIGHT_W, TOP_H,
               edge=PURPLE, face=BG_PANEL, lw=1.2, radius=0.006, z=0)

    glance_pad_x = 0.018
    glance_text_x = RIGHT_X + glance_pad_x
    glance_chip_x = RIGHT_X + glance_pad_x
    glance_chip_w = RIGHT_W - (2 * glance_pad_x)

    fig.patches.append(
        mpatches.FancyBboxPatch(
            (glance_chip_x, TOP_Y + TOP_H - 0.060),
            glance_chip_w,
            0.038,
            boxstyle='round,pad=0.002,rounding_size=0.004',
            transform=fig.transFigure,
            fc=PURPLE_MID,
            ec=PURPLE,
            lw=1.0,
            zorder=3
        )
    )

    fig.text(glance_text_x + 0.006, TOP_Y + TOP_H - 0.048,
             'PERFORMANCE SUMMARY', fontsize=18, fontweight='bold', color=WHITE)

    # Wider panel + controlled font sizes to avoid cropping
    fig.text(glance_text_x, TOP_Y + 0.265, 'BEST WEEK',
             fontsize=18, color=WHITE, fontweight='bold')
    fig.text(glance_text_x, TOP_Y + 0.232, f'WEEK {weeks[best_idx]}',
             fontsize=18, color=GREEN, fontweight='bold')

    fig.text(glance_text_x, TOP_Y + 0.183, 'PEAK OUTPUT',
             fontsize=18, color=WHITE, fontweight='bold')
    fig.text(glance_text_x, TOP_Y + 0.151, sci(ys[best_idx]),
             fontsize=18, color=WHITE, fontweight='bold')

    fig.text(glance_text_x, TOP_Y + 0.103, f'W1 → W{LATEST_WEEK} CHANGE',
             fontsize=16, color=WHITE, fontweight='bold')
    fig.text(glance_text_x, TOP_Y + 0.074, change_label(ys[0], ys[-1]),
             fontsize=18, color=s_col, fontweight='bold')

    fig.text(glance_text_x, TOP_Y + 0.051, 'STATUS',
             fontsize=16, color=WHITE, fontweight='bold')
    fig.text(glance_text_x, TOP_Y + 0.031, status,
             fontsize=17, color=s_col, fontweight='bold')

    # Week cards: one horizontal row, side by side cards
    cards_y = 0.180
    card_h = 0.300
    gap_x = CARD_GAP_X
    left = CARD_LEFT
    cols = CARD_COLS
    card_w = CARD_W
    for j, wk in enumerate(weeks):
        x = left + j*(card_w+gap_x)
        draw_week_card(fig, fn, wk, x, cards_y, card_w, card_h, ys)

    plan, _ = FINAL_QUERY_NOTE.get(fn, ('Adaptive refinement', ''))
    q = get_final_query(fn)
    qstr = '  –  '.join([f'{v:.6f}' for v in q])
    q_wrapped = '\n'.join(textwrap.wrap(qstr, width=95))
    draw_panel(fig, 0.012, 0.030, 0.976, 0.112, edge=PURPLE, face=BG_PANEL, lw=1.2, radius=0.006, z=0)
    fig.text(0.030, 0.112, 'PLANNED STRATEGY', fontsize=21, color=PURPLE, fontweight='bold')
    fig.text(0.175, 0.112, plan.replace('\n',''), fontsize=21, color=WHITE, va='center')
    fig.text(0.030, 0.068, 'RESULTS', fontsize=21, color=PURPLE, fontweight='bold')
    fig.text(0.175, 0.068, q_wrapped, fontsize=21, color=WHITE, fontweight='bold', family='monospace', va='center')

    path = os.path.join(VISUAL_OUTPUT_PATH, f'function_{fn:02d}_dashboard.png')
    safe_savefig(fig, path, dpi=180)

# ----------------------------
# Overview page
# ----------------------------

def draw_overview_card(fig, fn, x, y, w, h):
    weeks, xs, ys = get_fn_inputs_outputs(fn)
    status = status_from_series(ys)
    col = status_color(status)
    draw_panel(fig, x, y, w, h, edge=BORDER, face=BG_PANEL, lw=1.2, radius=0.006, z=0)
    fig.text(x+0.020*w, y+h-0.050, f'F{fn:02d}', fontsize=22, fontweight='bold', color=WHITE)
    fig.text(x+w-0.120, y+h-0.048, status, fontsize=20, fontweight='bold', color=col,
             bbox=dict(boxstyle='round,pad=0.25', fc=BG_CARD2, ec=col, lw=1.0))
    best = np.max(ys)
    #fig.text(x+0.290*w, y+h-0.084, f'Best: {sci(best)}', fontsize=17, color=GOLD, fontweight='bold')
    ax = fig.add_axes([x+0.012*w, y+0.235*h, w*0.965, h*0.640], projection='3d', zorder=2)
    draw_3d_bars(ax, ys, weeks=weeks, compact=True, fn=fn)
    fig.text(x+0.50*w, y+0.135*h, f'Best Week: Week {weeks[highest_week_index(ys, fn=fn)]}',
             fontsize=22, color=GOLD, fontweight='bold', ha='center')
    ch = change_label(ys[0], ys[-1])
    fig.text(x+0.50*w, y+0.075*h, ch, fontsize=23, color=col, fontweight='bold', ha='center')
    fig.text(x+0.50*w, y+0.025*h, f'Change (W1 → W{LATEST_WEEK})', fontsize=22, color=WHITE, ha='center')

def plot_executive_overview():
    fig = plt.figure(figsize=(38, 23), facecolor=BG_PAGE)
    add_header(fig, fn=None, subtitle=f'{LATEST_WEEK}-WEEK OPTIMISATION SUMMARY    |    STRATEGY: ADAPTIVE BAYESIAN')
    add_colour_legend(fig)
    fig.text(0.030, 0.860, 'WEEKLY PROGRESS — ALL FUNCTIONS', fontsize=25, color=CYAN, fontweight='bold')
    # Card grid aligned with the header/title box
    CONTENT_LEFT = 0.010
    CONTENT_RIGHT = 0.990
    CONTENT_W = CONTENT_RIGHT - CONTENT_LEFT

    top_y = 0.480
    gap_x = 0.015
    gap_y = 0.036
    card_w = (CONTENT_W - 3 * gap_x) / 4
    card_h = 0.348
    left = CONTENT_LEFT
    for idx, fn in enumerate(range(1,9)):
        r = idx // 4
        c = idx % 4
        x = left + c*(card_w+gap_x)
        y = top_y - r*(card_h+gap_y)
        draw_overview_card(fig, fn, x, y, card_w, card_h)
    path = os.path.join(VISUAL_OUTPUT_PATH, 'Weekly_progress_overview.png')
    safe_savefig(fig, path, dpi=180)

# ----------------------------
# Final applied strategy table helper
# ----------------------------
def build_applied_strategy_table():
    """Create a clean strategy-only table for the final dashboard outputs."""
    rows = []
    for fn in range(1, 9):
        row = {
            'Function': f'F{fn:02d}',
            'Use Case': FUNCTION_LABELS.get(fn, f'Function {fn}')
        }
        for wk in sorted(WEEKLY_RESULTS.keys()):
            row[f'Week {wk} Applied Strategy'] = STRATEGY_TEXT.get(fn, {}).get(wk, 'Adaptive Bayesian strategy').replace('\n', ' ')
        row['Final Strategy Note'] = FINAL_QUERY_NOTE.get(fn, ('Adaptive refinement',''))[0].replace('\n', ' ')
        rows.append(row)
    return pd.DataFrame(rows)

def plot_final_applied_strategy_table():
    """Dedicated final output page: strategy applied only, no result/score columns."""
    applied_strategy_df = build_applied_strategy_table()
    csv_path = os.path.join(VISUAL_OUTPUT_PATH, f'applied_strategy_table_week_{LATEST_WEEK}.csv')
    applied_strategy_df.to_csv(csv_path, index=False)

    fig = plt.figure(figsize=(62, 34), facecolor=BG_PAGE)
    add_header(fig, fn=None, subtitle='FINAL OUTPUT TABLE    |    APPLIED STRATEGY ONLY')
    fig.text(0.022, 0.865, 'FINAL APPLIED STRATEGY TABLE', fontsize=38, color=CYAN, fontweight='bold')

    visual_df = applied_strategy_df.copy()
    visual_df.columns = [c.replace(' Applied Strategy', '') for c in visual_df.columns]

    ax = fig.add_axes([0.020, 0.080, 0.960, 0.750])
    ax.axis('off')

    table = ax.table(
        cellText=visual_df.values,
        colLabels=visual_df.columns,
        loc='center',
        cellLoc='left',
        colLoc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)

    ncols = len(visual_df.columns)
    fixed_w = [0.070, 0.095]
    last_w = 0.105
    remaining = max(1.0 - sum(fixed_w) - last_w, 0.0)
    col_widths = fixed_w + [remaining / (ncols - 3)] * (ncols - 3) + [last_w]

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor(BORDER)
        cell.set_linewidth(0.75)
        cell.PAD = 0.060
        if c < len(col_widths):
            cell.set_width(col_widths[c])

        raw_txt = cell.get_text().get_text()
        if r == 0:
            cell.set_facecolor(BG_CARD2)
            cell.set_text_props(color=CYAN, weight='bold', ha='center', va='center', fontsize=21)
            cell.PAD = 0.035
        else:
            cell.set_facecolor(BG_CARD if r % 2 else BG_PANEL)
            if c == 0:
                cell.set_text_props(color=CYAN, weight='bold', ha='center', va='center', fontsize=21)
            else:
                wrap_width = 15 if c not in [1, ncols - 1] else 18
                cell.get_text().set_text('\n'.join(textwrap.wrap(raw_txt, width=wrap_width)))
                cell.set_text_props(color=WHITE, ha='left', va='center', fontsize=18)

    path = os.path.join(VISUAL_OUTPUT_PATH, 'final_applied_strategy_table.png')
    safe_savefig(fig, path, dpi=180)
    print(f'Saved applied strategy CSV -> {csv_path}')

# ----------------------------
# Strategy summary page
# ----------------------------
def plot_strategy_summary_page():
    """Strategy table page with fixed-width columns and readable wrapping."""
    fig = plt.figure(figsize=(62, 34), facecolor=BG_PAGE)
    add_header(fig, fn=None, subtitle='ACTUAL STRATEGY APPLIED ACROSS COMPLETED WEEKS')
    fig.text(0.025, 0.865, 'STRATEGY FOLLOWED EACH WEEK', fontsize=38, color=CYAN, fontweight='bold')

    cols = ['Function'] + [f'Week {w}' for w in sorted(WEEKLY_RESULTS.keys())] + ['Final Strategy Note']
    rows = []
    for fn in range(1, 9):
        row = [f'F{fn:02d}']
        for wk in sorted(WEEKLY_RESULTS.keys()):
            row.append(STRATEGY_TEXT.get(fn, {}).get(wk, 'Adaptive strategy').replace('\n', ' '))
        row.append(FINAL_QUERY_NOTE.get(fn, ('Adaptive refinement', ''))[0].replace('\n', ' '))
        rows.append(row)

    ax = fig.add_axes([0.025, 0.225, 0.950, 0.610])
    ax.axis('off')

    table = ax.table(
        cellText=rows,
        colLabels=cols,
        loc='center',
        cellLoc='left',
        colLoc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)

    ncols = len(cols)
    first_w = 0.055
    last_w = 0.135
    remaining = max(1.0 - first_w - last_w, 0.0)
    col_widths = [first_w] + [remaining / (ncols - 2)] * (ncols - 2) + [last_w]

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor(BORDER)
        cell.set_linewidth(0.75)
        cell.PAD = 0.060
        if c < len(col_widths):
            cell.set_width(col_widths[c])

        raw_txt = cell.get_text().get_text()
        if r == 0:
            cell.set_facecolor(PURPLE_MID)
            cell.set_text_props(color=WHITE, weight='bold', ha='center', va='center', fontsize=21)
            cell.PAD = 0.035
        else:
            cell.set_facecolor(BG_CARD if r % 2 else BG_PANEL)
            if c == 0:
                cell.set_text_props(color=CYAN, weight='bold', ha='center', va='center', fontsize=21)
            else:
                wrap_width = 13 if c < ncols - 1 else 20
                cell.get_text().set_text('\n'.join(textwrap.wrap(raw_txt, width=wrap_width)))
                cell.set_text_props(color=WHITE, ha='left', va='center', fontsize=18)


    logic = [
        ('Weeks 1–2', 'Baseline and broad exploration to identify responsive regions.'),
        ('Weeks 3–4', 'Local exploitation begins for promising functions.'),
        ('Weeks 5–6', 'Correction phase: rollback where needed and exploit improved basins.'),
        ('Final Note', 'Protect strong basins, micro-tune stable functions, and avoid unsupported extrapolation.'),
    ]

    y0 = 0.060
    card_h = 0.125
    x0 = 0.025
    gap = 0.014
    total_w = 0.950
    card_summary_w = (total_w - 3 * gap) / 4
    for i, (title, body) in enumerate(logic):
        x = x0 + i * (card_summary_w + gap)
        edge_col = PURPLE if i == 3 else BORDER
        draw_panel(fig, x, y0, card_summary_w, card_h, edge=edge_col, face=edge_col, alpha=0.18, lw=1.2, radius=0.006)
        fig.text(x + 0.012, y0 + 0.082, title, fontsize=22, color=PURPLE if i == 3 else CYAN, fontweight='bold')
        fig.text(x + 0.012, y0 + 0.030, '\n'.join(textwrap.wrap(body, width=42)), fontsize=19, color=WHITE, linespacing=1.16)

    path = os.path.join(VISUAL_OUTPUT_PATH, 'strategy_followed_each_week.png')
    safe_savefig(fig, path, dpi=180)

# ----------------------------
# Run all visual pages
# ----------------------------
def run_visual_dashboard():
    print('\n' + '='*70)
    print('Generating BBO dashboard visuals')
    print('='*70)
    os.makedirs(VISUAL_OUTPUT_PATH, exist_ok=True)
    plot_executive_overview()
    plot_strategy_summary_page()
    plot_final_applied_strategy_table()
    for fn in range(1, 9):
        plot_function_dashboard(fn)
    print('\nVisuals saved to:')
    print(VISUAL_OUTPUT_PATH)
    print('='*70 + '\n')

run_visual_dashboard()